# TRIAGE-EG Targeted Cross-Asset Survey v0.2

**This is a targeted cross-asset contract survey, not a complete Data Audit.**

Bootstrap có thể dùng internet để clone/fetch GitHub. Bản thân survey không gọi internet, không decode video, không đọc image pixels, không chạy model/GPU và không traversal tổng quát dataset.

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
DATASET_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
OUTPUT_ROOT = Path(os.environ.get("AIC_CROSS_ASSET_OUTPUT_ROOT", "/kaggle/working/cross_asset_survey_v02"))
REFRESH_REPO = os.environ.get("AIC_REFRESH_REPO", "0") == "1"
print("repo URL:", REPO_URL)
print("repo ref:", REPO_REF)
print("repo directory:", REPO_DIR)
print("dataset:", DATASET_ROOT)
print("output:", OUTPUT_ROOT)

## Repository bootstrap

Rerun-safe: clone khi repo chưa có; giữ detached `HEAD` hiện tại khi rerun; chỉ fetch lại khi đặt `AIC_REFRESH_REPO=1`. Repo private cần Git credential/Kaggle secret được cấu hình bên ngoài notebook.

In [ ]:
def run_git(*args: str, cwd: Path | None = None) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        ["git", *args], cwd=cwd, check=True, text=True, capture_output=True
    )

if (REPO_DIR / ".git").is_dir():
    print("Using existing repository checkout.")
    if REFRESH_REPO:
        print("Refreshing requested ref from origin...")
        run_git("fetch", "--depth", "1", "origin", REPO_REF, cwd=REPO_DIR)
        run_git("checkout", "--detach", "FETCH_HEAD", cwd=REPO_DIR)
    else:
        print("Keeping the existing detached HEAD; no network refresh requested.")
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"REPO_DIR exists but is not a Git checkout: {REPO_DIR}")
else:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    print("Cloning repository...")
    run_git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR))
    run_git("fetch", "--depth", "1", "origin", REPO_REF, cwd=REPO_DIR)
    run_git("checkout", "--detach", "FETCH_HEAD", cwd=REPO_DIR)

resolved_commit = run_git("rev-parse", "HEAD", cwd=REPO_DIR).stdout.strip()
print("resolved git commit:", resolved_commit)

In [ ]:
module_path = REPO_DIR / "src" / "triage_eg" / "data" / "cross_asset_survey.py"
if not module_path.is_file():
    raise RuntimeError(
        "cross_asset_survey.py is missing from the selected Git ref. "
        "Commit/push the v0.2 files, then set AIC_REPO_REF to that branch or commit. "
        f"Selected ref={REPO_REF!r}, commit={resolved_commit!r}."
    )
src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("survey module:", module_path)
print("python:", platform.python_version())

In [ ]:
from triage_eg.data.cross_asset_survey import (
    CrossAssetLimits, survey_cross_assets, write_outputs,
)

preferred_ids = ["L21_V017", "L24_V010", "L26_V215", "L28_V019", "L30_V025"]
limits = CrossAssetLimits(
    max_videos=5, max_object_json_total=25,
    max_object_json_bytes=1_048_576, max_mapping_rows=10_000, seed=2026,
)
result = survey_cross_assets(
    DATASET_ROOT, limits=limits, video_ids=preferred_ids, strict_root=True,
)
artifact_paths = write_outputs(result, OUTPUT_ROOT)
print(result.summary["disclaimer"])

## Asset ID-set comparison

In [ ]:
result.summary["id_set_comparison"]

## Selected sample videos

In [ ]:
{"selected": result.summary["selected_video_ids"], "reasons": result.summary["selection_reasons"]}

## Count alignment

In [ ]:
[{key: record[key] for key in (
    "video_id", "mapping_row_count", "clip_row_count",
    "keyframe_image_count", "object_json_count",
)} for record in result.records]

## Keyframe and Object filename alignment

In [ ]:
[{"video_id": record["video_id"],
  "keyframe": record["keyframe_filename_contract"],
  "clip": record["clip_row_contract"],
  "object": record["object_filename_contract"]} for record in result.records]

## Object JSON schema

In [ ]:
result.summary["object_schema_summary"]

## Duplicate frame_idx cases

In [ ]:
result.summary["duplicate_frame_idx_case_studies"]

## Verified, inferred, and unknown contracts

In [ ]:
{"verified": result.summary["verified_contracts"],
 "inferred": result.summary["inferred_contracts"],
 "unknown": result.summary["unknown_contracts"],
 "readiness": result.summary["next_stage_readiness"]}

In [ ]:
print("This is a targeted cross-asset contract survey, not a complete Data Audit.")
for name, path in artifact_paths.items():
    print(f"{name}: {path}")

## Download one result bundle

Chạy cell cuối sau khi survey hoàn tất. ZIP chỉ chứa năm artifact report của v0.2, không chứa dataset nguồn.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

required_artifacts = [
    OUTPUT_ROOT / "cross_asset_survey_v02.json",
    OUTPUT_ROOT / "cross_asset_survey_v02.md",
    OUTPUT_ROOT / "cross_asset_records.jsonl",
    OUTPUT_ROOT / "object_schema_samples.jsonl",
    OUTPUT_ROOT / "issues.jsonl",
]
missing_artifacts = [path for path in required_artifacts if not path.is_file()]
if missing_artifacts:
    raise RuntimeError(
        "Survey chưa sinh đủ artifact: "
        + ", ".join(str(path) for path in missing_artifacts)
    )

bundle_path = OUTPUT_ROOT.parent / "cross_asset_survey_v02_bundle.zip"
with ZipFile(bundle_path, mode="w", compression=ZIP_DEFLATED) as archive:
    for artifact_path in required_artifacts:
        archive.write(artifact_path, arcname=artifact_path.name)

print("Download this single file:")
print(bundle_path)
print("Included:", [path.name for path in required_artifacts])